## Notebook 04: Feature Engineering

### What this notebook does
Builds the feature store from the cleaned master dataframe.
Every feature is computed correctly within store-item groups
to prevent data leakage.

### Features we build
1. Lag features      — what happened 1, 7, 14, 28 days ago
2. Rolling features  — recent average and volatility
3. Time features     — day of week, month, weekend flag
4. External features — oil price momentum, promotion flag

### Critical rule
Every lag and rolling feature MUST use groupby(store_nbr, item_nbr)
Computing on the full dataframe mixes data across different products
and stores — this is data leakage and will inflate your metrics.

In [1]:
import pandas as pd
import numpy as np
import psutil
print("Ready")

Ready


In [2]:
# ── LOAD ──────────────────────────────────────────────────────────
df = pd.read_parquet("../data/processed/master_sample.parquet")

df["store_nbr"]   = df["store_nbr"].astype("int8")
df["item_nbr"]    = df["item_nbr"].astype("int32")
df["onpromotion"] = df["onpromotion"].astype("int8")
df["is_holiday"]  = df["is_holiday"].astype("int8")
df["perishable"]  = df["perishable"].astype("int8")
df["cluster"]     = df["cluster"].astype("int8")
df["unit_sales"]  = df["unit_sales"].astype("float32")
df["oil_price"]   = df["oil_price"].astype("float32")
for col in ["family","city","state","type"]:
    df[col] = df[col].astype("category")

df = df.sort_values(
    ["store_nbr","item_nbr","date"]
).reset_index(drop=True)
print(f"1. Loaded: {df.shape} | "
      f"RAM: {psutil.virtual_memory().available/1e9:.1f}GB free")

# ── LAG FEATURES ──────────────────────────────────────────────────
def add_date_lag(df, lag_days):
    lag_df = df[["store_nbr","item_nbr",
                 "date","unit_sales"]].copy()
    lag_df["date"] = lag_df["date"] + pd.Timedelta(days=lag_days)
    lag_df = lag_df.rename(
        columns={"unit_sales": f"lag_{lag_days}"}
    )
    return df.merge(
        lag_df, on=["store_nbr","item_nbr","date"], how="left"
    )

for lag in [1, 7, 14, 28]:
    df = add_date_lag(df, lag)
    df[f"lag_{lag}"] = df[f"lag_{lag}"].fillna(0).astype("float32")
    print(f"   lag_{lag} done")
print(f"2. Lags done | "
      f"RAM: {psutil.virtual_memory().available/1e9:.1f}GB free")

# ── ROLLING FEATURES ──────────────────────────────────────────────
df = df.sort_values(
    ["store_nbr","item_nbr","date"]
).reset_index(drop=True)

grp     = df.groupby(["store_nbr","item_nbr"])["unit_sales"]
shifted = grp.shift(1)

df["rolling_mean_7"] = (
    shifted.transform(
        lambda x: x.rolling(7, min_periods=1).mean()
    )
).astype("float32")

df["rolling_mean_28"] = (
    shifted.transform(
        lambda x: x.rolling(28, min_periods=1).mean()
    )
).astype("float32")

df["rolling_std_7"] = (
    shifted.transform(
        lambda x: x.rolling(7, min_periods=1).std().fillna(0)
    )
).astype("float32")
print(f"3. Rolling done | "
      f"RAM: {psutil.virtual_memory().available/1e9:.1f}GB free")

# ── TIME FEATURES ─────────────────────────────────────────────────
df["day_of_week"]  = df["date"].dt.dayofweek.astype("int8")
df["month"]        = df["date"].dt.month.astype("int8")
df["is_weekend"]   = (
    df["date"].dt.dayofweek.isin([5,6])
).astype("int8")
df["day_of_month"] = df["date"].dt.day.astype("int8")
df["week_of_year"] = (
    df["date"].dt.isocalendar().week.astype("int8")
)
print("4. Time features done")

# ── EXTERNAL FEATURES ─────────────────────────────────────────────
df["oil_lag7"] = df["oil_price"].shift(7).astype("float32")
df["oil_price_change_pct"] = (
    (df["oil_price"] - df["oil_lag7"]) /
    (df["oil_lag7"] + 1e-8)
).astype("float32")
df["discount_depth"] = df["onpromotion"].astype("int8")
df.drop(columns=["oil_lag7"], inplace=True)
print("5. External features done")

# ── TARGET ENCODING ───────────────────────────────────────────────
family_store_mean = (
    df.groupby(["family","store_nbr"])["unit_sales"]
    .mean()
    .reset_index()
    .rename(columns={"unit_sales": "family_store_mean"})
)
family_store_mean["family_store_mean"] = (
    family_store_mean["family_store_mean"].astype("float32")
)
df = df.merge(
    family_store_mean, on=["family","store_nbr"], how="left"
)
print("6. Target encoding done")

# ── FINAL FEATURE LIST ────────────────────────────────────────────
FEATURE_COLS = [
    "lag_1","lag_7","lag_14","lag_28",
    "rolling_mean_7","rolling_mean_28","rolling_std_7",
    "day_of_week","month","is_weekend",
    "day_of_month","week_of_year",
    "oil_price","oil_price_change_pct","discount_depth",
    "is_holiday","cluster","perishable",
    "family_store_mean"
]
TARGET = "unit_sales"

# Drop first 28 days — lag_28 is unreliable before this
first_date = df["date"].min() + pd.Timedelta(days=28)
df_model   = df[df["date"] >= first_date].copy()

# Null check
null_check = df_model[FEATURE_COLS].isnull().sum()
nulls_found = null_check[null_check > 0]

print(f"\n{'='*45}")
print(f"df shape:          {df.shape}")
print(f"Model-ready rows:  {df_model.shape}")
print(f"Features:          {len(FEATURE_COLS)}")
print(f"Nulls in features: "
      f"{'None — all clean' if len(nulls_found)==0 else nulls_found}")
print(f"RAM free:          "
      f"{psutil.virtual_memory().available/1e9:.1f}GB")
print(f"{'='*45}")

# ── SAVE ──────────────────────────────────────────────────────────
df.to_parquet(
    "../data/features/feature_store.parquet", index=False
)
df_model.to_parquet(
    "../data/features/model_ready.parquet", index=False
)
print("7. Saved both parquet files")
print("   → data/features/feature_store.parquet")
print("   → data/features/model_ready.parquet")
print("\nFeature engineering complete.")

1. Loaded: (28234961, 14) | RAM: 5.5GB free
   lag_1 done
   lag_7 done
   lag_14 done
   lag_28 done
2. Lags done | RAM: 5.4GB free
3. Rolling done | RAM: 4.3GB free
4. Time features done
5. External features done


C:\Users\innso\AppData\Local\Temp\ipykernel_5292\559143618.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["family","store_nbr"])["unit_sales"]


6. Target encoding done

df shape:          (28234961, 29)
Model-ready rows:  (27952032, 29)
Features:          19
Nulls in features: None — all clean
RAM free:          5.7GB
7. Saved both parquet files
   → data/features/feature_store.parquet
   → data/features/model_ready.parquet

Feature engineering complete.



```
28,234,961 total rows
27,952,032 model-ready rows (dropped first 28 days per series)
19 features — zero nulls
Saved to parquet — ready for modeling
```

The 282,929 dropped rows are the first 28 days of each store-item series where lag_28 would have been unreliable. That's correct behavior.


## Quick Summary — What Each Feature Captures


```
LAG FEATURES
lag_1   → what sold yesterday (very short term signal)
lag_7   → what sold same day last week (weekly seasonality)
lag_14  → two weeks ago (bi-weekly pattern)
lag_28  → four weeks ago (monthly pattern)

ROLLING FEATURES
rolling_mean_7   → average demand last 7 days (recent trend)
rolling_mean_28  → average demand last 28 days (medium term baseline)
rolling_std_7    → demand volatility last 7 days (uncertainty signal)

TIME FEATURES
day_of_week  → weekly shopping pattern (confirmed by decomposition)
month        → annual seasonality
is_weekend   → weekend demand spike
day_of_month → paycheck effects (sales spike around 1st and 15th)
week_of_year → holiday weeks, back-to-school, etc.

EXTERNAL FEATURES
oil_price            → economic conditions (confirmed -0.600 correlation)
oil_price_change_pct → momentum signal (rising vs falling oil)
discount_depth       → promotion active flag

BUSINESS FEATURES
is_holiday         → 11.4% demand uplift confirmed in EDA
cluster            → store group (similar stores behave similarly)
perishable         → affects demand volatility and zero patterns
family_store_mean  → baseline demand level for this product-store combo
```

---

## Where You Are Now

```
✓ Day 1 — Setup + data understood
✓ Day 2 — Data cleaned and merged
✓ Day 3 — EDA complete, 5 findings
✓ Day 4 — Feature store built, 19 features, zero nulls
→ Day 5 — Baseline models (starts now)
```
